Version | Authors
------------ | -------------
0.2 | Dennis McNab, Benjamin Dilly, Anton Kisel

Introduction
============
This is a interactive notebook regarding "Introduction to path planning". It's objective is to benchmark a random path smoothing algorithm against a deterministic path smoothing approach according to Bechthold and Glavina. The comparison focuses on the qualitative and quantitative effects of both methods.

Description:
* Used Robot: Point robot
* Benchmark tasks: trap, maze
* Planners: BasicPRM, VisibilityPRM, and LazyPRM

Imports
===========

In [ ]:
import planner_config
import planning
import diagram

In [ ]:
import IPTestSuitePointRobot as ts

from shapely.geometry import Point
from shapely import plotting

import matplotlib.pylab as plt
import matplotlib

Set-up of the test scenario and the configuration for all planner
===================================

Following is a procedure to compare the smoothing algorithms:

1. Configuration for every planner with **benchmark_config.plannerFactory**
2. The configuration and the planner are stored in the variable **plannerFactory**
3. The resulting setup variable is used as a unified interface to execute
   planning calls consistently across all planners.


In [ ]:
plannerFactory = planner_config.plannerFactory

In [ ]:
fullBenchList = ts.benchList
for benchmark in fullBenchList:
    print(benchmark.name)

# Planning and smoothing

Transfer of **plannerFactory** and the list of benchmark tasks to the planning function for pathplanning and subsequent smoothing of the paths

In [ ]:
testList = fullBenchList
resultList = planning.planning(plannerFactory, testList)

# Visualization

Visualization of benchmark tasks

In [ ]:
for benchmark in fullBenchList:
    fig_local = plt.figure(figsize=(7,7))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    ax.set_title(title)
    ax.set_xlim(benchmark.collisionChecker.getEnvironmentLimits()[0])
    ax.set_ylim(benchmark.collisionChecker.getEnvironmentLimits()[1])
    plotting.plot_points(Point(benchmark.startList[0]).buffer(.3), color="g", ax=ax)
    plotting.plot_points(Point(benchmark.goalList[0]).buffer(.3), color="b", ax=ax)
    benchmark.collisionChecker.drawObstacles(ax)

Visualization of the planned paths and the smoothed paths

In [ ]:
matplotlib.rcParams['animation.embed_limit'] = 256

for result in resultList:
    if result.solution != []:
        print(f"Animations {result.benchmark.name} {result.plannerFactoryName}")
        # Random Smoothing
        result.smoothing.visualize_smoothing(title=f"Path with Random smoothing\n [{result.plannerFactoryName}]")
        result.smoothing.animate_path(result.solution, title=f"Smoothed Path", maintitle=f"Random Smoothing\nBenchmark: {result.benchmark.name}\nPlanner: [{result.plannerFactoryName}]")
        # Smoothing with BG
        result.bg_smoother.visualize_smoothing(title=f"Path with Bechthold-Glavina smoothing\n [{result.plannerFactoryName}]")
        result.bg_smoother.animate_path(result.solution, title=f"Smoothed Path", maintitle=f"Bechthold-Glavina Smoothing\nBenchmark: {result.benchmark.name}\nPlanner: [{result.plannerFactoryName}]")
    else:
        print(f"{result.plannerFactoryName} {result.benchmark.name} no Path found")

# Evaluation

Diagrams showing the evaluation metrics: path length, number of nodes in the graph, planning and smoothing time

In [ ]:
diagram.generate_diagramm(testList, resultList)

Comparison of the path length and number of nodes for the two smoothing methods

In [ ]:
diagram.generate_timeplot(resultList)

# 5 times planning with LazyPRM
No animation, as the focus is on the diagrams and not on visualizing the paths.

In [ ]:
import IPLazyPRM
import IPVISLazyPRM

The same configurations for all planners to ensure that a path can be found

In [ ]:
plannerFactory2 = dict()
lazyConfig = dict()
lazyConfig["initialRoadmapSize"] = 40
lazyConfig["updateRoadmapSize"]  = 10
lazyConfig["kNearest"] = 15
lazyConfig["maxIterations"] = 50
plannerFactory2["lazyPRM"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM2"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM3"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM4"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM5"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]

Planning and smoothing

In [ ]:
resultList_lazy = planning.planning(plannerFactory2, testList)

Evaluation

In [ ]:
diagram.generate_diagramm(testList, resultList_lazy)

In [ ]:
diagram.generate_timeplot(resultList_lazy)